# Paper R4: splits 2–5 (tuning, re-evaluation, test evaluation) and baselines on all 5 splits

Conference paper, Phase 5 (decisions D33, D35, D36; claude/paper/phase4_design.md sections 2, 4, 9, 10). **Runs on the PC only** (XGBoost is fitted only on the PC; CatBoost thread_count = 12).

What it does, in this order:
1. **Checks the inputs:** data hash, Stage 4 freeze, `test_predictions.csv`, the Stage 1 index files against `MANIFEST.csv`.
2. **Builds and freezes the index files for splits 2–5** before any model is fitted (Stage 1 procedure: `train_test_split(pool, train_size=n, stratify=...)`; seeds from the 15th S2 stream `paper_outer`). A rerun verifies them byte for byte and never overwrites them.
3. **Split 1 reproduction check:** the six frozen thesis configurations are refitted on the thesis development rows and must reproduce `test_predictions.csv` exactly; otherwise the run stops before any new split.
4. **Baselines on split 1** (Altamimi settings with 0.50 and Youden thresholds, Ahmed reading = default LR, majority class).
5. **Splits 2–5**, per split: for LR, XGB, LGBM, MLP, RF and CAT, a search over the same 30 frozen configurations (5-fold CV, AP; MLP on the split's 50k subsample, D14), then a 5-fold re-evaluation on the full development folds (OOF probabilities, mean development AP, thresholds as in S12c), a refit on all development rows, and test predictions; then the baselines.

**Resumable:** each (split, model) and each baseline block is saved with a `done.json` written last. If the run stops, Run All again: finished blocks are skipped.

**Run:** set `R4_CONFIRMED = True` in Cell 2, put the notebook in `thesis_xai/`, keep the laptop on mains power with sleep disabled, and Run All. Expected time: about 4.9 h (up to about 7.3 h with thermal throttling). It can run across several sessions. Cell 6 prints the plan.

**Send back:** everything in `paper_r4_outputs/` except the `pipelines/` folders. Keep those on the PC for R5–R6.

In [1]:
# Cell 1 - environment guard (S10): before numpy / scikit-learn are imported.
import os, sys
TEST_MODE = os.environ.get("R4_TEST_MODE") == "1"          # Claude's workspace test only; never set on the PC
PROJECT_DIR = os.environ.get("R4_PROJECT_DIR", os.path.abspath("."))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
try:
    import env_setup
    env_setup.ensure_pinned(PROJECT_DIR, install=False)
except ImportError:
    if not TEST_MODE:
        raise SystemExit("env_setup.py not found in the project folder.")

All pinned packages present at the pinned versions.


In [2]:
# Cell 2 - configuration
R4_CONFIRMED = True                       # set True to run (D35/D36 approved)
if TEST_MODE:
    R4_CONFIRMED = True
if not R4_CONFIRMED:
    raise SystemExit("Set R4_CONFIRMED = True in Cell 2, then Run All.")

import hashlib, json, platform, time, warnings
from datetime import datetime, timezone
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss, f1_score,
                             matthews_corrcoef, roc_auc_score, roc_curve)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
import thesis_common as tc

FREEZE_SHA256 = "1dd5bf06708d148399675d9fc657857df1c86f015cbd83c24e79e8754b48c0d8"   # S4-047
PRED_SHA256 = "b9813461cd03e4552b50c245c8750648d08b9c48c1960d249642982bc15114ed"     # S5-136
MODELS6 = ["LR", "XGB", "LGBM", "MLP", "RF", "CAT"]                  # run order within a split (cheap first)
SPLITS = [int(s) for s in os.environ.get("R4_SPLITS", "2,3,4,5").split(",")]
CAT_THREADS = 12                                                     # D35 item 1
SEARCH_JOBS = {"LR": 6}                                              # S9: single-threaded LR searches folds in parallel
SUB_N = 50000                                                        # D14
N_BG, N_BG100, N_CORE = 500, 100, 500
REEVAL_TOL = 1e-4                                                    # S12a
TEST_FRACTION = float(os.environ.get("R4_TEST_FRACTION", "1.0")) if TEST_MODE else 1.0

# Seeds: the first 14 S2 streams are unchanged; paper_outer is the 15th (D33); per-split streams below.
SS15 = np.random.SeedSequence(tc.MASTER_SEED).generate_state(15)
assert [int(v) for v in SS15[:13]] == [tc.SEEDS[k] for k in tc.SEED_STREAMS], "seed derivation changed"
assert int(SS15[13]) == 3957024168, "14th stream (shap_permutation, S15k) changed"
PAPER_OUTER = int(SS15[14])
SPLIT_STREAMS = ["partition", "folds", "subsample_50k", "shap_background", "shap_perm_background", "test_core", "eval_bootstrap"]
def split_seeds(k):
    return {n: int(v) for n, v in zip(SPLIT_STREAMS, np.random.SeedSequence([PAPER_OUTER, k]).generate_state(len(SPLIT_STREAMS)))}

OUT = os.path.join(PROJECT_DIR, "paper_r4_TEST" if TEST_MODE else "paper_r4_outputs")
os.makedirs(OUT, exist_ok=True)
MACHINE = "workspace-test" if TEST_MODE else "pc"
if not TEST_MODE and (platform.system() != "Windows" or os.cpu_count() != 12):
    raise SystemExit("R4 runs on the project PC only (Windows, 12 logical CPUs): XGBoost results differ between machines (P1-008).")
print(f"machine={MACHINE} python={platform.python_version()} sklearn={sklearn.__version__} paper_outer={PAPER_OUTER}")
for k in SPLITS:
    print(f"split {k} seeds:", split_seeds(k))

machine=pc python=3.13.0 sklearn=1.6.1 paper_outer=2128606514
split 2 seeds: {'partition': 1175812276, 'folds': 1928346733, 'subsample_50k': 2695588701, 'shap_background': 2046412443, 'shap_perm_background': 4038824602, 'test_core': 3943562800, 'eval_bootstrap': 2424220024}
split 3 seeds: {'partition': 53387441, 'folds': 2243936840, 'subsample_50k': 421107203, 'shap_background': 1455755608, 'shap_perm_background': 4284788987, 'test_core': 3951056151, 'eval_bootstrap': 1600468367}
split 4 seeds: {'partition': 241135546, 'folds': 1797926918, 'subsample_50k': 3096314353, 'shap_background': 2815086372, 'shap_perm_background': 3930389921, 'test_core': 2191480547, 'eval_bootstrap': 2125626925}
split 5 seeds: {'partition': 2097939031, 'folds': 1886747120, 'subsample_50k': 3766643917, 'shap_background': 330525642, 'shap_perm_background': 3349837425, 'test_core': 139835216, 'eval_bootstrap': 3269299794}


In [3]:
# Cell 3 - inputs: data, freeze, Stage 5 predictions, Stage 1 index files (split 1)
def sha(path):
    return tc.sha256_file(path)

def find_data():
    for d in [os.path.join(PROJECT_DIR, "data"), PROJECT_DIR]:
        for n in ["diabetes_binary_health_indicators_BRFSS2015.csv", "diabetes_binary_health_indicators_BRFSS2015 1.csv",
                  "diabetes_binary_health_indicators_BRFSS2015_1.csv"]:
            if os.path.exists(os.path.join(d, n)):
                return os.path.join(d, n)
    raise SystemExit("dataset not found in data/ or the project folder")

DATA = find_data()
if sha(DATA) != tc.DATA_SHA256:
    raise SystemExit("dataset is not the canonical file (SHA-256)")
df = pd.read_csv(DATA)
X_all, y_all = df[tc.FEATURES], df[tc.TARGET].astype(int).to_numpy()
FZ_PATH = os.path.join(PROJECT_DIR, "stage4_outputs", "frozen_models.json")
if sha(FZ_PATH) != FREEZE_SHA256:
    raise SystemExit("frozen_models.json is not the Stage 4 freeze")
FZ = json.load(open(FZ_PATH, encoding="utf-8"))
PRED_PATH = os.path.join(PROJECT_DIR, "stage5_outputs", "test_predictions.csv")
if sha(PRED_PATH) != PRED_SHA256:
    raise SystemExit("stage5_outputs/test_predictions.csv is missing or not the Stage 5 file")
PRED = pd.read_csv(PRED_PATH, float_precision="round_trip")
IDX1 = os.path.join(PROJECT_DIR, "stage1_outputs", "indices")
man1 = pd.read_csv(os.path.join(IDX1, "MANIFEST.csv")).set_index("file").sha256
for f in ["dev_folds.csv", "dev_subsample_50k.csv"]:
    if sha(os.path.join(IDX1, f)) != man1[f]:
        raise SystemExit(f"{f} differs from MANIFEST.csv")
d1 = pd.read_csv(os.path.join(IDX1, "dev_folds.csv"))
DEV1, FOLD1 = d1.row_id.to_numpy(), d1.fold.to_numpy()
TEST1 = np.setdiff1d(np.arange(len(y_all)), DEV1)
assert np.array_equal(TEST1, PRED.row_id.to_numpy()) and np.array_equal(PRED.y.to_numpy(), y_all[TEST1])
CFG_ID = {m: FZ["models"][m]["config_id"] for m in MODELS6}
for m in MODELS6:                                                   # frozen parameters = the sampled configuration
    cfg = tc.sampled_configs(m)[CFG_ID[m]]
    for key, v in FZ["models"][m]["params_repr"].items():
        assert repr(cfg[key]) == v or str(cfg[key]) == v, (m, key)
print("inputs OK:", len(DEV1), "development /", len(TEST1), "test rows (split 1)")

inputs OK: 202944 development / 50736 test rows (split 1)


In [4]:
# Cell 4 - helpers: atomic writes, checkpoints, pipelines, metrics, thresholds (S12c)
def write_bytes(path, blob):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        f.write(blob)
    os.replace(tmp, path)

def csv_bytes(frame, digits=17):
    return frame.to_csv(index=False, lineterminator="\n", float_format=f"%.{digits}g").encode("utf-8")

def write_or_verify(path, blob):
    # frozen files are created once; a rerun must reproduce them byte for byte (S5)
    if os.path.exists(path):
        if open(path, "rb").read() != blob:
            raise SystemExit(f"{path} exists and differs from the rebuilt file; it is never overwritten. Report this.")
        return "verified"
    write_bytes(path, blob)
    return "created"

def done_path(*parts):
    return os.path.join(OUT, *parts, "done.json")

def is_done(*parts):
    p = done_path(*parts)
    if not os.path.exists(p):
        return False
    rec = json.load(open(p, encoding="utf-8"))
    for rel, h in rec["files"].items():
        if sha(os.path.join(OUT, rel)) != h:
            raise SystemExit(f"{rel} differs from its done.json record; delete that block's folder and rerun.")
    return True

def mark_done(parts, rel_files, extra=None):
    rec = {"files": {r: sha(os.path.join(OUT, r)) for r in rel_files}, "machine": MACHINE,
           "finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), **(extra or {})}
    write_bytes(done_path(*parts), json.dumps(rec, indent=1).encode("utf-8"))

def make(m):
    return tc.make_pipeline(m, n_jobs=CAT_THREADS if m == "CAT" else -1)

def configured(m, cid):
    p = make(m); p.set_params(**tc.sampled_configs(m)[cid]); return p

def predict(pipe, m, X):
    if m == "RF":
        pipe.set_params(clf__n_jobs=1)                  # S12e: single-threaded RF prediction
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=DeprecationWarning)
        return pipe.predict_proba(X)[:, 1]

def at(yy, p, t):
    pred = (p >= t).astype(int)
    return {"threshold": t, "sensitivity": pred[yy == 1].mean(), "specificity": 1 - pred[yy == 0].mean(),
            "MCC": matthews_corrcoef(yy, pred), "precision": pred[yy == 1].sum() / max(pred.sum(), 1),
            "F1": f1_score(yy, pred, zero_division=0), "accuracy": accuracy_score(yy, pred)}

def thresholds(yy, p):
    # S12c: candidates = distinct OOF values, positive if p >= t. MCC-max (ties: higher sensitivity, then closer
    # to 0.50, then lower t); screening = highest t with sensitivity >= 0.80
    order = np.argsort(-p, kind="stable"); ps, ys = p[order], yy[order]
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[np.flatnonzero(ps[1:] != ps[:-1]), len(ps) - 1]      # last index of each distinct value
    t, TP, FP = ps[last], tp[last].astype(float), fp[last].astype(float)
    P, N = ys.sum(), len(ys) - ys.sum(); FN, TN = P - TP, N - FP
    den = np.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
    mcc = np.where(den > 0, (TP * TN - FP * FN) / np.where(den > 0, den, 1), 0.0)
    sens = TP / P
    best = max(range(len(t)), key=lambda i: (round(mcc[i], 15), sens[i], -abs(t[i] - 0.5), -t[i]))
    ok = np.flatnonzero(sens >= 0.80)
    return float(t[best]), float(t[ok].max())

def point_metrics(yy, p, t_mcc, t_scr):
    r = {"AP": average_precision_score(yy, p), "ROC_AUC": roc_auc_score(yy, p), "Brier": brier_score_loss(yy, p)}
    for name, t in [("050", 0.5), ("mcc", t_mcc), ("screen", t_scr)]:
        r.update({f"{k}_{name}": v for k, v in at(yy, p, t).items()})
    return r

In [5]:
# Cell 5 - index files for splits 2-5 (built and frozen before any model is fitted)
ROWS_ALL = np.arange(len(y_all))
if TEST_MODE and TEST_FRACTION < 1:
    ROWS_ALL = np.sort(train_test_split(ROWS_ALL, train_size=TEST_FRACTION, stratify=y_all, random_state=0)[0])

def samp(pool, n, seed, strat=None):
    s = y_all[pool] if strat is None else strat
    return np.sort(np.asarray(pool)[train_test_split(np.arange(len(pool)), train_size=n, stratify=s, random_state=seed)[0]])

SPLIT_IDX, man_rows = {}, []
for k in SPLITS:
    s = split_seeds(k)
    dev, test = train_test_split(ROWS_ALL, test_size=0.2, stratify=y_all[ROWS_ALL], random_state=s["partition"])
    dev, test = np.sort(dev), np.sort(test)
    fold = np.empty(len(dev), int)
    for j, (_, v) in enumerate(StratifiedKFold(5, shuffle=True, random_state=s["folds"]).split(dev, y_all[dev])):
        fold[v] = j
    n_sub = SUB_N if not TEST_MODE else min(SUB_N, len(dev) // 4)
    sub_pos = np.sort(train_test_split(np.arange(len(dev)), train_size=n_sub, stratify=y_all[dev] * 5 + fold,
                                       random_state=s["subsample_50k"])[0])
    bg = samp(dev, N_BG, s["shap_background"]); bg100 = samp(bg, N_BG100, s["shap_perm_background"])
    core = samp(test, N_CORE, s["test_core"])
    files = {"dev_folds.csv": pd.DataFrame({"row_id": dev, "fold": fold, "y": y_all[dev]}),
             "dev_subsample_50k.csv": pd.DataFrame({"row_id": dev[sub_pos], "fold": fold[sub_pos], "y": y_all[dev[sub_pos]]}),
             "test_rows.csv": pd.DataFrame({"row_id": test, "y": y_all[test]}),
             "shap_background_500_dev.csv": pd.DataFrame({"row_id": bg, "y": y_all[bg]}),
             "shap_perm_background_100_dev.csv": pd.DataFrame({"row_id": bg100, "y": y_all[bg100]}),
             "test_core_500.csv": pd.DataFrame({"row_id": core, "y": y_all[core]})}
    assert len(np.intersect1d(dev, test)) == 0 and set(bg100) <= set(bg) and set(core) <= set(test)
    for f, fr in files.items():
        blob = fr.to_csv(index=False, lineterminator="\n").encode("utf-8")
        status = write_or_verify(os.path.join(OUT, "indices", f"split{k}", f), blob)
        man_rows.append({"split": k, "file": f, "rows": len(fr), "positives": int(fr.y.sum()), "sha256": hashlib.sha256(blob).hexdigest(), "status": status})
    SPLIT_IDX[k] = {"dev": dev, "fold": fold, "test": test, "sub_pos": sub_pos}
MAN = pd.DataFrame(man_rows)
mblob = MAN.drop(columns="status").to_csv(index=False, lineterminator="\n").encode("utf-8")
mpath = os.path.join(OUT, "indices", "MANIFEST_splits" + "_".join(map(str, SPLITS)) + ".csv")
print("MANIFEST:", write_or_verify(mpath, mblob))
print(MAN.to_string(index=False))

MANIFEST: created
 split                             file   rows  positives                                                           sha256  status
     2                    dev_folds.csv 202944      28277 b3dafc5bd5bbef4db3eb06b826109d321c08bc9d00d39add9f926e01873d92eb created
     2            dev_subsample_50k.csv  50000       6965 85e3529b911fecd4a1f5f27165d08900f94a40251206ccd919dbcc2017698256 created
     2                    test_rows.csv  50736       7069 f7a734f83a7c5e4f2fa0fa9ae6fc89a0a35b5e3ab887547135fe0dd80ead0c63 created
     2      shap_background_500_dev.csv    500         70 61300b8c027732ecd761864d6b5ba35632283e607ecbb161d8c0cbfdd6142f8f created
     2 shap_perm_background_100_dev.csv    100         14 b6581876b2f67ceea0b544038b90ef5c6aaff730c9eb0e2dce9a40ca3e41b1ca created
     2                test_core_500.csv    500         70 7c62cd902c65fc2a5f1cd13678cee83a1837ae456abdd32faaa4ed0af1b42502 created
     3                    dev_folds.csv 202944      28277 90505fe

In [6]:
# Cell 6 - plan (PC planning estimates from S3-023-S3-030, S7-001-S7-010; not results)
SEARCH_MIN = {"LR": 3.79, "XGB": 7.17, "LGBM": 7.56, "MLP": 7.55, "RF": 17.72, "CAT": 22.98}
FIT_S = {"LR": 1.24, "XGB": 4.94, "LGBM": 3.02, "MLP": 14.58, "RF": 8.35, "CAT": 14.32}
def planned_min(k, m):
    return SEARCH_MIN[m] + 6 * FIT_S[m] / 60          # search + 5 re-evaluation fits + final refit
BASE_MIN = 1.5
plan = [("split1", "reproduction", sum(FIT_S.values()) / 60, is_done("split1", "reproduction")),
        ("split1", "baselines", BASE_MIN, is_done("split1", "baselines"))]
for k in SPLITS:
    plan += [(f"split{k}", m, planned_min(k, m), is_done(f"split{k}", m)) for m in MODELS6]
    plan.append((f"split{k}", "baselines", BASE_MIN, is_done(f"split{k}", "baselines")))
PLAN = pd.DataFrame(plan, columns=["split", "block", "planned_min", "done"])
print(PLAN.round(1).to_string(index=False))
left = PLAN.loc[~PLAN.done, "planned_min"].sum()
print(f"Remaining: about {left / 60:.1f} h (up to about {1.5 * left / 60:.1f} h with thermal throttling).")

 split        block  planned_min  done
split1 reproduction          0.8 False
split1    baselines          1.5 False
split2           LR          3.9 False
split2          XGB          7.7 False
split2         LGBM          7.9 False
split2          MLP          9.0 False
split2           RF         18.6 False
split2          CAT         24.4 False
split2    baselines          1.5 False
split3           LR          3.9 False
split3          XGB          7.7 False
split3         LGBM          7.9 False
split3          MLP          9.0 False
split3           RF         18.6 False
split3          CAT         24.4 False
split3    baselines          1.5 False
split4           LR          3.9 False
split4          XGB          7.7 False
split4         LGBM          7.9 False
split4          MLP          9.0 False
split4           RF         18.6 False
split4          CAT         24.4 False
split4    baselines          1.5 False
split5           LR          3.9 False
split5          XGB      

In [7]:
# Cell 7 - split 1: the six frozen configurations must reproduce test_predictions.csv before any new split runs
if not is_done("split1", "reproduction"):
    rows = []
    for m in MODELS6:
        p = predict(configured(m, CFG_ID[m]).fit(X_all.iloc[DEV1], y_all[DEV1]), m, X_all.iloc[TEST1])
        d = np.abs(p - PRED[f"p_{m}"].to_numpy())
        rows.append({"model": m, "config_id": CFG_ID[m], "max_abs_diff": float(d.max()), "rows_differing": int((d > 0).sum()),
                     "AP": average_precision_score(y_all[TEST1], p)})
        print(rows[-1], flush=True)
    R = pd.DataFrame(rows)
    # PC: exact equality (S7-052: refits on the PC reproduce Stage 5 exactly). Workspace test only: last-bit
    # differences (<= 1e-6) are tolerated and XGB is not checked, because it differs between machines (P1-008).
    bad = R[R.max_abs_diff > 0] if not TEST_MODE else R[(R.max_abs_diff > 1e-6) & (R.model != "XGB")]
    write_bytes(os.path.join(OUT, "split1", "reproduction", "reproduction.csv"), csv_bytes(R))
    if len(bad):
        raise SystemExit(f"Split 1 reproduction failed for {list(bad.model)}; nothing else runs. Report this.")
    mark_done(["split1", "reproduction"], ["split1/reproduction/reproduction.csv"])
print(pd.read_csv(os.path.join(OUT, "split1", "reproduction", "reproduction.csv")).to_string(index=False))

{'model': 'LR', 'config_id': 20, 'max_abs_diff': 0.0, 'rows_differing': 0, 'AP': np.float64(0.40014707631979407)}
{'model': 'XGB', 'config_id': 2, 'max_abs_diff': 0.0, 'rows_differing': 0, 'AP': np.float64(0.43047341472070716)}
{'model': 'LGBM', 'config_id': 19, 'max_abs_diff': 0.0, 'rows_differing': 0, 'AP': np.float64(0.4319233849276689)}
{'model': 'MLP', 'config_id': 21, 'max_abs_diff': 0.0, 'rows_differing': 0, 'AP': np.float64(0.42615691948541645)}
{'model': 'RF', 'config_id': 6, 'max_abs_diff': 0.0, 'rows_differing': 0, 'AP': np.float64(0.4277190198717933)}
{'model': 'CAT', 'config_id': 7, 'max_abs_diff': 0.0, 'rows_differing': 0, 'AP': np.float64(0.43189492328076606)}
model  config_id  max_abs_diff  rows_differing       AP
   LR         20             0               0 0.400147
  XGB          2             0               0 0.430473
 LGBM         19             0               0 0.431923
  MLP         21             0               0 0.426157
   RF          6             0      

In [8]:
# Cell 8 - baselines (D33 item 5, D35 item 3): retrained on the split's development rows, scored on its test rows
scaled = lambda est: Pipeline([("scale", StandardScaler()), ("clf", est)])
BASELINES = {
    "ALT_LR": lambda: scaled(LogisticRegression(max_iter=1000, class_weight="balanced")),
    "ALT_DT": lambda: DecisionTreeClassifier(max_depth=8, random_state=tc.MODEL_SEED),
    "ALT_RF": lambda: RandomForestClassifier(n_estimators=200, max_depth=12, n_jobs=-1, random_state=tc.MODEL_SEED),
    "ALT_XGB": lambda: XGBClassifier(n_estimators=250, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                                     tree_method="hist", n_jobs=-1, random_state=tc.MODEL_SEED, verbosity=0),
    "AHMED_LR_default": lambda: scaled(LogisticRegression()),        # Ahmed et al. (stated reading) = default-parameter LR
    "MAJORITY": lambda: DummyClassifier(strategy="most_frequent")}

def proba(est, X):
    if isinstance(est, RandomForestClassifier):
        est.set_params(n_jobs=1)
    return est.predict_proba(X)[:, 1]

def run_baselines(k, dev, fold, test):
    tag = f"split{k}"
    if is_done(tag, "baselines"):
        return
    Xd, yd, Xt, yt = X_all.iloc[dev].to_numpy(), y_all[dev], X_all.iloc[test].to_numpy(), y_all[test]
    rows, preds = [], {"row_id": test, "y": yt}
    for name, mk in BASELINES.items():
        p = proba(mk().fit(Xd, yd), Xt); preds[f"p_{name}"] = p
        r = {"split": k, "baseline": name, "thr_rule": "0.50", "AP": average_precision_score(yt, p), "ROC_AUC": roc_auc_score(yt, p), **at(yt, p, 0.5)}
        rows.append(r)
        if name.startswith("ALT"):
            oof = np.zeros(len(yd))
            for j in range(5):
                tr, va = fold != j, fold == j
                oof[va] = proba(mk().fit(Xd[tr], yd[tr]), Xd[va])
            fpr, tpr, thr = roc_curve(yd, oof); t = float(thr[np.argmax(tpr - fpr)])
            rows.append({**r, "thr_rule": "Youden (OOF dev)", **at(yt, p, t)})
        print({k_: (round(v, 4) if isinstance(v, float) else v) for k_, v in rows[-1].items()}, flush=True)
    write_bytes(os.path.join(OUT, tag, "baselines", "baselines_metrics.csv"), csv_bytes(pd.DataFrame(rows), 10))
    write_bytes(os.path.join(OUT, tag, "baselines", "baselines_test_predictions.csv"), csv_bytes(pd.DataFrame(preds)))
    mark_done([tag, "baselines"], [f"{tag}/baselines/baselines_metrics.csv", f"{tag}/baselines/baselines_test_predictions.csv"])

run_baselines(1, DEV1, FOLD1, TEST1)

{'split': 1, 'baseline': 'ALT_LR', 'thr_rule': 'Youden (OOF dev)', 'AP': np.float64(0.3982), 'ROC_AUC': np.float64(0.8209), 'threshold': 0.4652, 'sensitivity': np.float64(0.7983), 'specificity': np.float64(0.6958), 'MCC': np.float64(0.3538), 'precision': np.float64(0.2982), 'F1': 0.4342, 'accuracy': 0.7101}
{'split': 1, 'baseline': 'ALT_DT', 'thr_rule': 'Youden (OOF dev)', 'AP': np.float64(0.3987), 'ROC_AUC': np.float64(0.8148), 'threshold': 0.1369, 'sensitivity': np.float64(0.772), 'specificity': np.float64(0.7062), 'MCC': np.float64(0.3449), 'precision': np.float64(0.2984), 'F1': 0.4304, 'accuracy': 0.7154}
{'split': 1, 'baseline': 'ALT_RF', 'thr_rule': 'Youden (OOF dev)', 'AP': np.float64(0.4276), 'ROC_AUC': np.float64(0.8259), 'threshold': 0.1465, 'sensitivity': np.float64(0.7878), 'specificity': np.float64(0.7086), 'MCC': np.float64(0.358), 'precision': np.float64(0.3044), 'F1': 0.4391, 'accuracy': 0.7196}
{'split': 1, 'baseline': 'ALT_XGB', 'thr_rule': 'Youden (OOF dev)', 'AP': n

In [9]:
# Cell 9 - one (split, model) block: search, re-evaluation, thresholds, final refit, test predictions
def run_model(k, m):
    tag, I = f"split{k}", SPLIT_IDX[k]
    if is_done(tag, m):
        return
    t0 = time.time()
    dev, fold, test, sub = I["dev"], I["fold"], I["test"], I["sub_pos"]
    rel = lambda f: f"{tag}/{m}/{f}"
    if m == "MLP":                                     # D14: tuned on the 50k subsample with its own folds
        Xs, ys, fs = X_all.iloc[dev[sub]], y_all[dev[sub]], fold[sub]
    else:
        Xs, ys, fs = X_all.iloc[dev], y_all[dev], fold
    cv = [(np.flatnonzero(fs != j), np.flatnonzero(fs == j)) for j in range(5)]
    search = RandomizedSearchCV(make(m), tc.SEARCH_SPACES[m], n_iter=30, scoring="average_precision", cv=cv, refit=False,
                                random_state=tc.SEEDS["search_sampling"], n_jobs=SEARCH_JOBS.get(m, 1), error_score="raise")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        search.fit(Xs, ys)
    cvr = search.cv_results_
    if [dict(p) for p in cvr["params"]] != tc.sampled_configs(m):
        raise SystemExit(f"{tag} {m}: evaluated configurations differ from the 30 frozen ones")
    cid = int(np.flatnonzero(cvr["rank_test_score"] == 1)[0])          # ties -> lowest config_id (S11a)
    n_ties = int((cvr["rank_test_score"] == 1).sum())
    cvdf = pd.DataFrame({"config_id": range(30), **{f"fold{j}_ap": cvr[f"split{j}_test_score"] for j in range(5)},
                         "mean_ap": cvr["mean_test_score"], "rank": cvr["rank_test_score"]})
    write_bytes(os.path.join(OUT, rel("cv_results.csv")), csv_bytes(cvdf))
    t_search = time.time() - t0
    # re-evaluation on the full development folds (S12a): OOF probabilities for all development rows
    Xd, yd = X_all.iloc[dev], y_all[dev]
    oof, fold_ap = np.zeros(len(dev)), []
    for j in range(5):
        tr, va = fold != j, fold == j
        oof[va] = predict(configured(m, cid).fit(Xd[tr], yd[tr]), m, Xd[va])
        fold_ap.append(average_precision_score(yd[va], oof[va]))
    if m != "MLP":
        dif = np.abs(np.array(fold_ap) - cvdf.loc[cid, [f"fold{j}_ap" for j in range(5)]].to_numpy(float)).max()
        if dif > REEVAL_TOL:
            raise SystemExit(f"{tag} {m}: re-evaluation does not reproduce the search fold APs (max diff {dif:.2e})")
    t_mcc, t_scr = thresholds(yd, oof)
    write_bytes(os.path.join(OUT, rel("oof.csv")), csv_bytes(pd.DataFrame({"row_id": dev, "fold": fold, "y": yd, "p": oof})))
    # final refit on all development rows, test predictions
    final = configured(m, cid).fit(Xd, yd)
    pipe_rel = f"pipelines/{tag}_{m}.joblib"
    os.makedirs(os.path.join(OUT, "pipelines"), exist_ok=True)
    joblib.dump(final, os.path.join(OUT, pipe_rel))
    p = predict(final, m, X_all.iloc[test]); yt = y_all[test]
    write_bytes(os.path.join(OUT, rel("test_predictions.csv")), csv_bytes(pd.DataFrame({"row_id": test, "y": yt, "p": p})))
    summ = {"split": k, "model": m, "config_id": cid, "rank1_ties": n_ties, "params": {kk: repr(v) for kk, v in tc.sampled_configs(m)[cid].items()},
            "search_mean_ap": float(cvdf.loc[cid, "mean_ap"]), "dev_mean_ap": float(np.mean(fold_ap)), "dev_sd_ap": float(np.std(fold_ap)),
            "dev_fold_ap": [float(a) for a in fold_ap], "threshold_mcc": t_mcc, "threshold_screen": t_scr,
            "test": {kk: float(v) for kk, v in point_metrics(yt, p, t_mcc, t_scr).items()},
            "pipeline_sha256": sha(os.path.join(OUT, pipe_rel)), "search_min": t_search / 60, "total_min": (time.time() - t0) / 60}
    write_bytes(os.path.join(OUT, rel("summary.json")), json.dumps(summ, indent=1).encode("utf-8"))
    mark_done([tag, m], [rel("cv_results.csv"), rel("oof.csv"), rel("test_predictions.csv"), rel("summary.json")])
    print(f"[{tag} {m}] config {cid}, dev AP {summ['dev_mean_ap']:.4f}, test AP {summ['test']['AP']:.4f}, "
          f"{summ['total_min']:.1f} min", flush=True)

In [10]:
# Cell 10 - run splits 2-5 (resumable)
t_all = time.time()
for k in SPLITS:
    for m in MODELS6:
        run_model(k, m)
    run_baselines(k, SPLIT_IDX[k]["dev"], SPLIT_IDX[k]["fold"], SPLIT_IDX[k]["test"])
    print(f"split {k} finished; session time {(time.time() - t_all) / 60:.1f} min", flush=True)

[split2 LR] config 16, dev AP 0.4043, test AP 0.4075, 8.9 min
[split2 XGB] config 24, dev AP 0.4354, test AP 0.4360, 9.9 min
[split2 LGBM] config 3, dev AP 0.4351, test AP 0.4340, 10.3 min
[split2 MLP] config 9, dev AP 0.4307, test AP 0.4339, 14.6 min
[split2 RF] config 9, dev AP 0.4301, test AP 0.4286, 28.4 min
[split2 CAT] config 7, dev AP 0.4364, test AP 0.4359, 34.6 min
{'split': 2, 'baseline': 'ALT_LR', 'thr_rule': 'Youden (OOF dev)', 'AP': np.float64(0.4061), 'ROC_AUC': np.float64(0.8207), 'threshold': 0.4573, 'sensitivity': np.float64(0.8041), 'specificity': np.float64(0.6841), 'MCC': np.float64(0.3476), 'precision': np.float64(0.2918), 'F1': 0.4282, 'accuracy': 0.7008}
{'split': 2, 'baseline': 'ALT_DT', 'thr_rule': 'Youden (OOF dev)', 'AP': np.float64(0.4058), 'ROC_AUC': np.float64(0.8162), 'threshold': 0.1348, 'sensitivity': np.float64(0.7827), 'specificity': np.float64(0.6987), 'MCC': np.float64(0.3456), 'precision': np.float64(0.2961), 'F1': 0.4296, 'accuracy': 0.7104}
{'spl

In [11]:
# Cell 11 - summary tables, top model per split (D7 rule), environment log
rows = []
for m in MODELS6:                                   # split 1 = thesis (Stage 4 freeze, Stage 5 predictions)
    p = PRED[f"p_{m}"].to_numpy(); yt = PRED.y.to_numpy(); fm = FZ["models"][m]
    rows.append({"split": 1, "model": m, "config_id": fm["config_id"], "dev_mean_ap": fm["dev_mean_ap"],
                 "threshold_mcc": fm["threshold_mcc"], "threshold_screen": fm["threshold_screen"],
                 **point_metrics(yt, p, fm["threshold_mcc"], fm["threshold_screen"])})
for k in SPLITS:
    for m in MODELS6:
        s = json.load(open(os.path.join(OUT, f"split{k}", m, "summary.json"), encoding="utf-8"))
        rows.append({"split": k, "model": m, "config_id": s["config_id"], "dev_mean_ap": s["dev_mean_ap"],
                     "threshold_mcc": s["threshold_mcc"], "threshold_screen": s["threshold_screen"], **s["test"]})
S = pd.DataFrame(rows)
S["top_model"] = S.groupby("split").dev_mean_ap.transform("max") == S.dev_mean_ap      # exact ties keep MODELS order
write_bytes(os.path.join(OUT, "r4_models_summary.csv"), csv_bytes(S, 10))
B = pd.concat([pd.read_csv(os.path.join(OUT, f"split{k}", "baselines", "baselines_metrics.csv")) for k in [1] + SPLITS])
write_bytes(os.path.join(OUT, "r4_baselines_summary.csv"), csv_bytes(B, 10))
print(S.pivot(index="model", columns="split", values="AP").round(4).to_string())
print("top model per split (highest mean development AP):", S[S.top_model].set_index("split").model.to_dict())
print(B[B.thr_rule == "0.50"].pivot(index="baseline", columns="split", values="AP").round(4).to_string())
from importlib import metadata
env = {"machine": MACHINE, "python": platform.python_version(), "platform": platform.platform(), "cpu_count": os.cpu_count(),
       "packages": {p: metadata.version(p) for p in ["numpy", "pandas", "scipy", "scikit-learn", "xgboost", "lightgbm", "catboost"]},
       "paper_outer": PAPER_OUTER, "split_seeds": {k: split_seeds(k) for k in SPLITS}, "test_mode": TEST_MODE,
       "written_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
write_bytes(os.path.join(OUT, f"env_log_r4_{MACHINE}.json"), json.dumps(env, indent=1).encode("utf-8"))
print("\nSend back: everything in", OUT, "except the pipelines/ folder (keep it on the PC for R5-R6).")

split       1       2       3       4       5
model                                        
CAT    0.4319  0.4359  0.4492  0.4411  0.4444
LGBM   0.4319  0.4340  0.4482  0.4403  0.4427
LR     0.4001  0.4075  0.4165  0.4071  0.4112
MLP    0.4262  0.4339  0.4445  0.4361  0.4396
RF     0.4277  0.4286  0.4408  0.4328  0.4368
XGB    0.4305  0.4360  0.4498  0.4408  0.4423
top model per split (highest mean development AP): {1: 'CAT', 2: 'CAT', 3: 'CAT', 4: 'CAT', 5: 'CAT'}
split                  1       2       3       4       5
baseline                                                
AHMED_LR_default  0.4001  0.4074  0.4165  0.4071  0.4112
ALT_DT            0.3987  0.4058  0.4153  0.4060  0.4094
ALT_LR            0.3982  0.4061  0.4153  0.4056  0.4092
ALT_RF            0.4276  0.4285  0.4414  0.4325  0.4358
ALT_XGB           0.4311  0.4349  0.4486  0.4421  0.4417
MAJORITY          0.1393  0.1393  0.1393  0.1393  0.1393

Send back: everything in C:\Users\USER\Downloads\thesis_xai\paper_r4_outp